<a href="https://colab.research.google.com/github/IvMatic/gpt2-logit-lens-experiments/blob/main/notebooks/01_logit_lens_basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformer-lens einops jaxtyping fancy-einsum -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 977.7/977.7 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 100.0 MB/s eta 0:00:00


In [3]:
import torch
import pandas as pd
import matplotlib.pyplot as plt

from transformer_lens import HookedTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded pretrained model gpt2-small into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint(name='hook_embed')
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint(name='hook_pos_embed')
  (blocks): ModuleList(
    (0): TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint(name='blocks.0.ln1.hook_scale')
        (hook_normalized): HookPoint(name='blocks.0.ln1.hook_normalized')
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint(name='blocks.0.ln2.hook_scale')
        (hook_normalized): HookPoint(name='blocks.0.ln2.hook_normalized')
      )
      (attn): Attention(
        (hook_k): HookPoint(name='blocks.0.attn.hook_k')
        (hook_q): HookPoint(name='blocks.0.attn.hook_q')
        (hook_v): HookPoint(name='blocks.0.attn.hook_v')
        (hook_z): HookPoint(name='blocks.0.attn.hook_z')
        (hook_attn_scores): HookPoint(name='blocks.0.attn.hook_attn_scores')
        (hook_pattern): HookPoint(name='blocks.0.attn.hook_pattern')
        (hook_result): HookPoint(name='blo

In [5]:
prompts = [
    "The capital of France is",
    "The opposite of hot is",
    "The sky is",
    "A dog says",
    "Two plus two is",
]

for prompt in prompts:
    tokens = model.to_tokens(prompt)
    logits = model(tokens)
    next_token_logits = logits[0, -1]
    top_tokens = torch.topk(next_token_logits, k=5).indices

    print("\nPROMPT:", prompt)
    for tok in top_tokens:
        print(model.to_string(tok), float(next_token_logits[tok]))


PROMPT: The capital of France is
 now 14.3899564743042
 the 14.152020454406738
 a 14.097540855407715
 home 13.960546493530273
 in 13.824111938476562

PROMPT: The opposite of hot is
 cold 14.218467712402344
 the 13.952598571777344
 not 13.646394729614258
 a 13.611642837524414
 hot 13.433122634887695

PROMPT: The sky is
 the 17.151775360107422
 falling 16.47016143798828
 blue 15.984930992126465
 a 15.930471420288086
 dark 15.13430118560791

PROMPT: A dog says
 it 15.897514343261719
 he 15.866292953491211
 she 15.621702194213867
 that 14.272146224975586
 his 13.88235855102539

PROMPT: Two plus two is
 a 14.82457160949707
 the 13.746610641479492
 not 13.586526870727539
 an 12.869592666625977
 one 12.373997688293457


/tmp/ipykernel_572/1158702661.py:17: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(model.to_string(tok), float(next_token_logits[tok]))
